# D2City as the **normal-bag pool** for DADA-2000 original — feasibility EDA

Plan: `.project/plans/katvad-d2city-normal-bag-eda.md` · Parent: `katvad-dada-original-corpus.md`

**Question.** With **positive bags = full-length DADA-2000 original videos** and
**negative bags = D2City dashcam clips**, do D2City negatives teach a linear reader of the
frozen CLIP features the **accident**, or the **source**? Same country + dashcam + similar fps
is necessary, not sufficient: `0_Normal_Driving` had all three and still leaked (C28, C32).

**GPU runtime** (T4 is enough). Nothing in `core/` changes; no model is trained.

| § | gate | bar (pre-registered, plan §6) |
|---|---|---|
| 1 | **G-I** inventory (HARD) | decodable ≥ 99 %, fps = 25 on ≥ 99 %, duration 29–31 s on ≥ 95 % |
| 3 | **G-L** length leak (HARD, chosen recipe) | clip-level length AUC ∈ [0.45, 0.55] |
| 4 | sanity | R0 `auc_macro` ∈ [0.62, 0.68] (≈ Gate D0 0.6518), else **stop: pipeline bug** |
| 4 | **G-X** (the gate) | PASS: X ≥ 0.60 and Δ(X−R0) ≥ −0.03 · FAIL: X < 0.55 or Δ < −0.06 |
| 4 | **G-M** | Δ(M−R0) ≥ −0.01 |
| 4 | G-S, shortcut | descriptive (S read against S-ref = DoTA; shortcut red flag > 0.90) |

Bring back: `outputs/D2City_eda/{eda_d2city.json, eda_d2city.md, montage.png, autocorr_seconds.png}`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE = '/content/drive/MyDrive/Thesis'
os.environ['PROJECT_ROOT']       = DRIVE
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'
for v in ('KATVAD_DATA_ROOT', 'KATVAD_CACHE_ROOT', 'KATVAD_OUTPUT_ROOT'):
    os.makedirs(os.environ[v], exist_ok=True)

os.environ['REPO'] = f'{DRIVE}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'eda' / 'protocol.py').is_file(), f'no core/ checkout at {REPO} -- sync it first'
os.environ['PYTHONPATH'] = str(REPO)
sys.path.insert(0, str(REPO))

# core.constants reads KATVAD_* at import time -> import after the block above.
from core import constants  # noqa: E402

# --- D2City (the candidate negative pool) ---------------------------------------------
D2_ROOT = constants.DATA_ROOT / 'D2City'
D2_ZIPS = D2_ROOT / 'training-video'           # 000{1..7}.zip, each '000N/<md5>.mp4'
D2_XML = D2_ROOT / 'training-annotation'       # 000N/<md5>.xml (dirs) or 000N.zip
# One cache dir per transform (C2). V0 = project transform (_ncc); V1 = 2.40:1 band first.
D2_CLIP = {'V0': constants.CLIP_CACHE_DIR / 'D2City_s7_ncc',
           'V1': constants.CLIP_CACHE_DIR / 'D2City_s7_ncc_ar240'}

# --- DADA-2000 original (positives) -- READ ONLY --------------------------------------
DADA_DATASET = constants.DADA_ORIGIN_DATASET                  # 'DADA2000_orig'
DADA_CLIP = constants.CLIP_CACHE_DIR / DADA_DATASET           # keyed by SOURCE clip, stride 8, ncc
DADA_T2 = constants.DATA_ROOT / DADA_DATASET                  # T2 corpus: meta.json carries on-disk
                                                              # total_frames + span per source
DADA_FRAMES = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME  # montage only

# --- DoTA (S-ref only; never trained on) -----------------------------------------------
DOTA_DATA = constants.DATA_ROOT / 'DoTA' / 'labels_s8'
DOTA_CLIP = constants.CLIP_CACHE_DIR / 'DoTA_s8_ncc'

OUT = constants.OUTPUT_ROOT / 'D2City_eda'                    # Drive: survives the runtime
WORK = Path('/content/d2city')                                # VM-local NVMe, never Drive
OUT.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

for name, path in (('D2City zips', D2_ZIPS), ('D2City xml', D2_XML), ('DADA clip', DADA_CLIP),
                   ('DADA T2 corpus', DADA_T2), ('DADA frames*', DADA_FRAMES),
                   ('DoTA data', DOTA_DATA), ('DoTA clip', DOTA_CLIP), ('out', OUT)):
    print(f'  {name:15s} {path}   {"OK" if path.exists() else "MISSING"}')
print('  (* optional: only the montage reads DADA frames)')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- section 2 will be very slow'
df -h /content | tail -1

### 0.1 Pre-registered constants (plan §6) — edit nothing here after the first run

Stride is **derived**, not chosen: DADA samples every 8 frames at 30 fps (assumption **A1**:
658,476 frames / 6.1 h in the DADA-2000 paper; the release ships PNGs so it cannot be
measured here) = 0.267 s. The D2City stride is the one closest to that time step at 25 fps.

In [ ]:
import json
import logging

import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
LOG = logging.getLogger('d2city_eda')

SEED = constants.SEED
DADA_FPS = 30.0                                  # A1 (literature, unverified)
D2_FPS = 25.0                                    # A2 (measured on 0001/, re-checked in section 1)
DADA_STRIDE = constants.FRAME_STRIDE             # 8 -- the DADA cache's stride, fixed (C2)
STRIDE_CANDIDATES = range(4, 13)
DADA_STEP_S = DADA_STRIDE / DADA_FPS
D2_STRIDE = min(STRIDE_CANDIDATES, key=lambda s: abs(s / D2_FPS - DADA_STEP_S))
V1_ASPECT = 1584 / 660                           # DADA native 2.40:1 (DADA_ORIGIN_PHASE0 3.3)

# G-I
GI_DECODABLE = 0.99
GI_FPS_SHARE = 0.99
GI_DURATION_S = (29.0, 31.0)
GI_DURATION_SHARE = 0.95
# G-L
GL_BAND = (0.45, 0.55)
PACK_FILL = 0.8                                  # L-match draws 80 % of D2City capacity (see make_segments)
TEST_MIXES = (0.0, 0.5, 1.0, 2.0)                # D2City bags per DADA video in a test set
# Probes
FOLDS = constants.EDA_PROBE_FOLDS                # 5
T95 = 2.776                                      # two-sided t, FOLDS - 1 = 4 df
R0_SANITY = (0.62, 0.68)
GX_PASS, GX_FAIL = 0.60, 0.55
GX_DELTA_PASS, GX_DELTA_MARGINAL = -0.03, -0.06
GM_DELTA = -0.01
GS_REF_MARGIN = 0.02
ARM_RULE_MARGIN = 0.02
SHORTCUT_RED = 0.90
# Mechanics (no effect on any number)
INVENTORY_WORKERS = 8
DECODE_WORKERS = 3
ENCODE_BATCH = 64
MONTAGE_N = 8

print(f'DADA step {DADA_STEP_S:.3f} s | D2City stride {D2_STRIDE} -> {D2_STRIDE / D2_FPS:.3f} s')
for s in (6, 7, 8):
    print(f'  stride {s}: {s / D2_FPS:.3f} s ({s / D2_FPS / DADA_STEP_S - 1:+.1%} vs DADA)')
assert D2_STRIDE == 7, 'the plan was written for stride 7 -- re-read plan P2 before continuing'

REPORT: dict = {'plan': '.project/plans/katvad-d2city-normal-bag-eda.md',
                'config': {'seed': SEED, 'dada_fps_assumed': DADA_FPS, 'd2_fps': D2_FPS,
                           'dada_stride': DADA_STRIDE, 'd2_stride': D2_STRIDE,
                           'v1_aspect': V1_ASPECT, 'folds': FOLDS,
                           'probe_C': constants.EDA_PROBE_C,
                           'probe_max_iter': constants.EDA_PROBE_MAX_ITER}}


def save_report() -> None:
    # Written after every section, so a dead runtime keeps what was measured.
    (OUT / 'eda_d2city.json').write_text(json.dumps(REPORT, indent=2, default=float))

## 1. Inventory — Gate **G-I** (HARD)

Unzip to VM-local disk, then count **files**, not folders (C10). Every clip is opened and its
first frame decoded; section 2 decodes all of it and logs any late failure.

In [ ]:
import zipfile

VIDEOS = WORK / 'videos'
VIDEOS.mkdir(exist_ok=True)
zips = sorted(D2_ZIPS.glob('*.zip'))
assert zips, f'no zip parts under {D2_ZIPS}'
for zp in zips:
    marker = VIDEOS / f'.{zp.stem}.done'
    if marker.exists():
        continue
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(VIDEOS)
    marker.touch()
    print(f'  unzipped {zp.name}')

paths = sorted(VIDEOS.rglob('*.mp4'))
ids = [p.stem for p in paths]
dupes = sorted({i for i in ids if ids.count(i) > 1})
assert not dupes, f'{len(dupes)} duplicate ids across parts (C26): {dupes[:5]}'
VIDEO_PATHS = dict(zip(ids, paths))
print(f'{len(zips)} parts -> {len(VIDEO_PATHS)} mp4 files')

In [ ]:
import csv
from concurrent.futures import ThreadPoolExecutor

import av


def probe(item: tuple[str, Path]) -> dict:
    vid, path = item
    row = {'id': vid, 'part': path.parent.name, 'ok': False}
    try:
        with av.open(str(path)) as container:
            stream = container.streams.video[0]
            fps = float(stream.average_rate) if stream.average_rate else float('nan')
            duration = (float(stream.duration * stream.time_base) if stream.duration
                        else float(container.duration) / av.time_base)
            frames = stream.frames or round(duration * fps)
            next(container.decode(stream))
        row.update(ok=True, fps=fps, width=stream.codec_context.width,
                   height=stream.codec_context.height, frames=int(frames),
                   duration_s=duration, codec=stream.codec_context.name)
    except (av.error.FFmpegError, StopIteration, OSError, ValueError) as exc:
        row['error'] = repr(exc)
    return row


with ThreadPoolExecutor(INVENTORY_WORKERS) as pool:
    INVENTORY = list(pool.map(probe, sorted(VIDEO_PATHS.items())))

with (OUT / 'inventory.csv').open('w', newline='') as fh:
    keys = ['id', 'part', 'ok', 'fps', 'width', 'height', 'frames', 'duration_s', 'codec', 'error']
    writer = csv.DictWriter(fh, fieldnames=keys)
    writer.writeheader()
    writer.writerows(INVENTORY)

good = [r for r in INVENTORY if r['ok']]
n = len(INVENTORY)
share_ok = len(good) / n
share_fps = sum(abs(r['fps'] - D2_FPS) < 0.01 for r in good) / n
share_dur = sum(GI_DURATION_S[0] <= r['duration_s'] <= GI_DURATION_S[1] for r in good) / n
res = {}
for r in good:
    res[f"{r['width']}x{r['height']}"] = res.get(f"{r['width']}x{r['height']}", 0) + 1
frame_counts = np.array([r['frames'] for r in good])
gi_pass = share_ok >= GI_DECODABLE and share_fps >= GI_FPS_SHARE and share_dur >= GI_DURATION_SHARE

REPORT['G-I'] = {'clips': n, 'decodable': share_ok, 'fps25': share_fps, 'duration_in_band': share_dur,
                 'resolution': res, 'frames_pct': dict(zip(map(str, constants.EDA_PERCENTILES),
                                                          np.percentile(frame_counts, constants.EDA_PERCENTILES).tolist())),
                 'failed': [r['id'] for r in INVENTORY if not r['ok']], 'pass': gi_pass}
save_report()
print(json.dumps({k: v for k, v in REPORT['G-I'].items() if k != 'failed'}, indent=1))
print(f"failed: {len(REPORT['G-I']['failed'])}")
assert gi_pass, 'G-I FAILED -- fix ingest (re-download the failing parts) before anything below'
print('G-I PASS')

### 1.1 Scene profile from the box tracks (descriptive)

DADA has no boxes, so this only describes D2City: how busy its scenes are and how much of the
traffic is pedestrians / two- and three-wheelers — the mix DADA's accidents involve.

In [ ]:
import xml.etree.ElementTree as ET

XML_DIR = WORK / 'xml'
if not XML_DIR.exists():
    XML_DIR.mkdir()
    for zp in sorted(D2_XML.glob('*.zip')):
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(XML_DIR)
xml_files = sorted(XML_DIR.rglob('*.xml')) or sorted(D2_XML.rglob('*.xml'))
VULNERABLE = {'person', 'bicycle', 'motorcycle', 'open-tricycle', 'closed-tricycle'}

per_clip_density, label_counts, covered = [], {}, 0
for xf in xml_files:
    if xf.stem not in VIDEO_PATHS:
        continue
    covered += 1
    root = ET.parse(xf).getroot()
    n_frames = int(root.findtext('meta/frames') or 0)
    boxes = 0
    for track in root.iter('track'):
        k = sum(1 for b in track.iter('box') if b.get('outside', '0') == '0')
        label_counts[track.get('label')] = label_counts.get(track.get('label'), 0) + k
        boxes += k
    if n_frames:
        per_clip_density.append(boxes / n_frames)

total = sum(label_counts.values()) or 1
dens = np.array(per_clip_density)
REPORT['xml'] = {'clips_with_xml': covered,
                 'objects_per_frame_pct': dict(zip(map(str, constants.EDA_PERCENTILES),
                                                   np.percentile(dens, constants.EDA_PERCENTILES).tolist())),
                 'label_share': {k: v / total for k, v in sorted(label_counts.items(), key=lambda kv: -kv[1])},
                 'vulnerable_share': sum(v for k, v in label_counts.items() if k in VULNERABLE) / total}
save_report()
print(json.dumps(REPORT['xml'], indent=1))

### 1.2 Montage — **look at it** (human check)

Row 1: D2City under **V0** (full frame squashed to 224², the project's `_ncc`).
Row 2: the same frames under **V1** (centre 2.40:1 band first). Row 3: DADA at 224² (if its
frames are on Drive). Look for timestamps, logos, hood, letterbox bars, colour cast — anything a
linear probe could read instead of the road.

In [ ]:
import random

import matplotlib.pyplot as plt
from PIL import Image

from core.data.video_io import list_frame_images, read_sampled_frames


def band_crop(frames: np.ndarray, aspect: float = V1_ASPECT) -> np.ndarray:
    # Centre horizontal band of `aspect` (W/H); a no-op on frames already that wide.
    height, width = frames.shape[1:3]
    band = round(width / aspect)
    if band >= height:
        return frames
    top = (height - band) // 2
    return frames[:, top:top + band]


def squash(frame: np.ndarray) -> np.ndarray:
    size = constants.CROP_SIZE
    return np.asarray(Image.fromarray(frame).resize((size, size), Image.BILINEAR))


rng = random.Random(SEED)
pick = rng.sample(sorted(VIDEO_PATHS), min(MONTAGE_N, len(VIDEO_PATHS)))
mid = []
for vid in pick:
    # frames 0, 350, 700 -> keep the middle one (~14 s in). Decodes the whole clip; 8 clips only.
    mid.append(read_sampled_frames(VIDEO_PATHS[vid], stride=50 * D2_STRIDE)[1])
rows = [[squash(f) for f in mid], [squash(band_crop(f[None])[0]) for f in mid]]
labels = ['D2City V0', 'D2City V1']
if DADA_FRAMES.exists():
    folders = sorted(DADA_FRAMES.glob(f'*/*/{constants.DADA_ORIGIN_IMAGES_SUBDIR}'))
    dada_row = []
    for folder in rng.sample(folders, min(MONTAGE_N, len(folders))):
        imgs = list_frame_images(folder)
        dada_row.append(squash(np.asarray(Image.open(imgs[len(imgs) // 2]).convert('RGB'))))
    rows.append(dada_row)
    labels.append('DADA (ncc)')
fig, axes = plt.subplots(len(rows), MONTAGE_N, figsize=(2 * MONTAGE_N, 2.2 * len(rows)))
for r, (row, label) in enumerate(zip(rows, labels)):
    for c in range(MONTAGE_N):
        ax = axes[r][c]
        ax.axis('off')
        if c < len(row):
            ax.imshow(row[c])
    axes[r][0].set_title(label, loc='left', fontsize=9)
fig.tight_layout()
fig.savefig(OUT / 'montage.png', dpi=110)
plt.show()

## 2. Extraction — D2City CLIP features, stride 7, arms V0 / V1

One decode per clip feeds both arms. Pixels are preprocessed **in batches** (a 1080p clip at
stride 7 is ~107 frames; preprocessing all of it at once is ~2.6 GB float32 — C9). Writes are
atomic and resumable (`save_array` / `is_complete`, C11), straight to Drive. DADA's cache is not
touched (C2).

In [ ]:
import time

import torch

from core.data.dataset_files import num_sampled_frames
from core.device import resolve_device
from core.tools.extract_clip_features import load_pretrained_encoder, preprocess_frames
from core.tools.feature_cache import is_complete, save_array

DEVICE = resolve_device('auto')
ENCODER = load_pretrained_encoder(DEVICE)
ARMS = {'V0': lambda f: f, 'V1': band_crop}
for d in D2_CLIP.values():
    d.mkdir(parents=True, exist_ok=True)


@torch.no_grad()
def encode(frames: np.ndarray) -> np.ndarray:
    chunks = []
    for start in range(0, len(frames), ENCODE_BATCH):
        pixels = preprocess_frames(frames[start:start + ENCODE_BATCH], constants.CROP_SIZE,
                                   center_crop=False).to(DEVICE)
        chunks.append(ENCODER(pixel_values=pixels).image_embeds.float().cpu())
    return torch.cat(chunks).numpy().astype(np.float32)


ok_ids = sorted(r['id'] for r in INVENTORY if r['ok'])
pending = [v for v in ok_ids if not all(is_complete(D2_CLIP[a] / f'{v}.npy') for a in ARMS)]
print(f'resume: {len(ok_ids) - len(pending)}/{len(ok_ids)} done, {len(pending)} to go')

failed, t0 = {}, time.time()
with ThreadPoolExecutor(DECODE_WORKERS) as pool:
    futures = {}
    queue = list(pending)
    while queue or futures:
        while queue and len(futures) < DECODE_WORKERS + 1:   # bounded RAM: ~0.7 GB per 1080p clip
            vid = queue.pop(0)
            futures[vid] = pool.submit(read_sampled_frames, VIDEO_PATHS[vid], D2_STRIDE)
        vid = next(iter(futures))
        try:
            frames = futures.pop(vid).result()
        except (av.error.FFmpegError, OSError, ValueError) as exc:
            failed[vid] = repr(exc)
            LOG.warning('decode failed %s: %s', vid, exc)
            continue
        for arm, transform in ARMS.items():
            save_array(D2_CLIP[arm] / f'{vid}.npy', encode(transform(frames)))
        done = len(pending) - len(queue) - len(futures)
        if done % 25 == 0:
            rate = done / (time.time() - t0)
            print(f'  {done}/{len(pending)}  {rate:.2f} clips/s  eta {(len(pending) - done) / max(rate, 1e-9) / 60:.0f} min')

manifest = {'d2_stride': D2_STRIDE, 'transform': 'no_center_crop', 'crop_size': constants.CROP_SIZE,
            'arms': {'V0': 'full frame', 'V1': f'centre band aspect {V1_ASPECT:.4f}'},
            'clip_model': constants.CLIP_MODEL_NAME, 'clip_revision': constants.CLIP_MODEL_REVISION,
            'clips_ok': len(ok_ids) - len(failed), 'decode_failed': failed}
for arm, d in D2_CLIP.items():
    (d / 'extraction_manifest.json').write_text(json.dumps(manifest, indent=2))
REPORT['extraction'] = manifest
save_report()
print(f'done; {len(failed)} late decode failures (logged, excluded)')

## 3. Load both sides, then the length / oracle **lever table** — Gate **G-L**

DADA labels are rebuilt **exactly as the T2 pipeline builds them**: `DadaRecord` + `sampled_frame_labels`
from the T2 corpus's own `meta.json` (on-disk `total_frames` and `normalized_span` per source) —
the label convention is imported, not restated. A source whose feature rows disagree with its label
length is dropped **with a count** (C2/C13).

In [ ]:
from core.data.dada import DadaRecord, sampled_frame_labels
from core.eda.corpus import describe, load_dataset_files

t2_meta = json.loads((DADA_T2 / constants.META_FILENAME).read_text())
sources: dict[str, dict] = {}
for wid, m in t2_meta.items():
    src = m.get('source', wid)
    if m.get('normalized_span') is not None:
        sources.setdefault(src, m)

DADA_FEATS, DADA_LABELS, mismatch, missing = {}, {}, [], []
for src, m in sorted(sources.items()):
    path = DADA_CLIP / f'{src}.npy'
    if not path.is_file():
        missing.append(src)
        continue
    record = DadaRecord(video_id=src, folder_name=src, class_name=m['class_name'],
                        fault_label=m['fault_label'], total_frames=int(m['total_frames']),
                        span=tuple(m['normalized_span']), accident_frac=m.get('accident_frac'))
    labels = np.asarray(sampled_frame_labels(record, DADA_STRIDE), dtype=np.int8)
    feats = np.load(path)
    if len(feats) != len(labels):
        mismatch.append(f'{src}: {len(feats)} rows vs {len(labels)} labels')
        continue
    if labels.any():
        DADA_FEATS[src], DADA_LABELS[src] = feats, labels

print(f'DADA sources {len(sources)} -> usable {len(DADA_FEATS)} | missing features {len(missing)} '
      f'| row mismatch {len(mismatch)}')
assert len(mismatch) <= 0.01 * len(sources), f'row mismatches {mismatch[:5]} -- wrong cache for these labels (C2)'
dada_len = np.array([len(v) for v in DADA_LABELS.values()])
print(describe(dada_len.tolist(), 'DADA sampled length'))

D2_FEATS = {arm: {p.stem: np.load(p) for p in sorted(d.glob('*.npy'))} for arm, d in D2_CLIP.items()}
assert set(D2_FEATS['V0']) == set(D2_FEATS['V1']), 'the two arms cover different clips'
d2_len = {v: len(a) for v, a in D2_FEATS['V0'].items()}
print(describe(list(d2_len.values()), 'D2City sampled length (whole clip)'))

DOTA_PRE = {}
try:
    dota_labels = load_dataset_files(DOTA_DATA, 'DoTA').frame_labels_test
except FileNotFoundError as exc:
    dota_labels = {}
    print(f'DoTA unavailable -> S-ref skipped ({exc})')
for vid, lab in dota_labels.items():
    lab = np.asarray(lab)
    path = DOTA_CLIP / f'{vid}.npy'
    if lab.any() and path.is_file():
        first = int(np.argmax(lab))
        feats = np.load(path)
        if first >= 1 and len(feats) == len(lab):
            DOTA_PRE[vid] = feats[:first]
print(f'DoTA clips with >= 1 pre-anomaly frame: {len(DOTA_PRE)}')
REPORT['data'] = {'dada_sources': len(sources), 'dada_usable': len(DADA_FEATS),
                  'dada_missing_features': len(missing), 'dada_row_mismatch': len(mismatch),
                  'dada_length': describe(dada_len.tolist(), 'DADA sampled length'),
                  'd2_clips': len(d2_len), 'd2_length': describe(list(d2_len.values()), 'D2City length'),
                  'dota_pre_clips': len(DOTA_PRE)}
save_report()

In [ ]:
from core.eda.protocol import clip_constant_oracle, clip_length_leak


def make_segments(recipe: str, clip_lengths: dict[str, int], dada_lengths: np.ndarray,
                  seed: int = SEED) -> list[tuple[str, str, int, int]]:
    # (segment_id, clip_id, start, end) -- non-overlapping, in sampled-frame units.
    # L-match draws ALL lengths first, then first-fit packs them into random clips with room.
    # Drawing-until-it-no-longer-fits (per clip) rejects long draws more often and biases the
    # bags SHORT -- measured on the dry run: length AUC 0.578 'shorter'. Packing at PACK_FILL
    # of capacity places ~every draw, so the placed set IS the drawn distribution.
    rng = np.random.default_rng(seed)
    out: list[tuple[str, str, int, int]] = []
    if recipe == 'L-raw':
        return [(clip, clip, 0, total) for clip, total in sorted(clip_lengths.items())]
    if recipe == 'L-fixed':
        fixed = int(np.median(dada_lengths))
        for clip, total in sorted(clip_lengths.items()):
            for k in range(total // fixed):
                out.append((f'{clip}__s{k}', clip, k * fixed, (k + 1) * fixed))
        return out
    fits = dada_lengths[dada_lengths <= max(clip_lengths.values())]
    n_draw = int(PACK_FILL * sum(clip_lengths.values()) / fits.mean())
    cursor = dict.fromkeys(clip_lengths, 0)
    clips = sorted(clip_lengths)
    for length in rng.choice(fits, n_draw):
        room = [c for c in clips if clip_lengths[c] - cursor[c] >= length]
        if not room:
            PACK_STATS['unplaced'] += 1
            continue
        clip = room[rng.integers(len(room))]
        out.append((f'{clip}__s{cursor[clip]}', clip, cursor[clip], cursor[clip] + int(length)))
        cursor[clip] += int(length)
    PACK_STATS.update(drawn=n_draw, placed=len(out),
                      dada_longer_than_d2_clip=float(np.mean(dada_lengths > max(clip_lengths.values()))))
    return out


def lever_row(segments, dada_labels: list[np.ndarray], mix: float, seed: int = SEED) -> dict:
    rng = np.random.default_rng(seed)
    n_neg = min(len(segments), round(mix * len(dada_labels)))
    chosen = [segments[i] for i in rng.choice(len(segments), n_neg, replace=False)] if n_neg else []
    labels = dada_labels + [np.zeros(e - s, dtype=np.int8) for _, _, s, e in chosen]
    row = {'mix': mix, 'd2_bags': n_neg, 'oracle_micro': clip_constant_oracle(labels)['auc_micro']}
    if n_neg:
        leak = clip_length_leak(labels)
        row.update(length_auc=leak['auc_clip_level'], length_micro=leak['auc_micro'],
                   direction=leak['direction'])
    return row


PACK_STATS = {'unplaced': 0}
dada_label_list = [DADA_LABELS[v] for v in sorted(DADA_LABELS)]
LEVER, SEGMENTS = {}, {}
for recipe in ('L-raw', 'L-fixed', 'L-match'):
    SEGMENTS[recipe] = make_segments(recipe, d2_len, dada_len)
    LEVER[recipe] = [lever_row(SEGMENTS[recipe], dada_label_list, m) for m in TEST_MIXES]
    print(f'{recipe:8s} {len(SEGMENTS[recipe]):5d} bags')
    for row in LEVER[recipe]:
        print('   ', {k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})

CHOSEN = 'L-match'
gl_auc = next(r['length_auc'] for r in LEVER[CHOSEN] if r['mix'] == 1.0)
gl_pass = GL_BAND[0] <= gl_auc <= GL_BAND[1]
print(f'L-match packing: {PACK_STATS}  (dada_longer_than_d2_clip = share of DADA videos no D2City clip can match)')
REPORT['G-L'] = {'lever': LEVER, 'packing': PACK_STATS, 'bags': {k: len(v) for k, v in SEGMENTS.items()},
                 'chosen': CHOSEN, 'length_auc_mix1': gl_auc, 'pass': gl_pass}
save_report()
print(f'G-L {CHOSEN}: length AUC {gl_auc:.4f} in {GL_BAND} -> {"PASS" if gl_pass else "FAIL"}')
assert gl_pass, 'G-L FAILED -- read the lever table; do NOT run the probes on a leaking recipe'

### 3.1 Time scale check (A1) — feature change per **second**, not per sampled step

If DADA really is 30 fps, stride 8 (DADA) and stride 7 (D2City) sample the world 0.267 s vs
0.280 s apart and the two curves below should overlap within their spread. A D2City curve well
**above** DADA's = its scenes change slower = a "static-ness" cue a model can read.

In [ ]:
from core.eda.features import temporal_autocorrelation

dada_pre = {v: DADA_FEATS[v][:int(np.argmax(DADA_LABELS[v]))] for v in DADA_FEATS}
dada_pre = {v: a for v, a in dada_pre.items() if len(a) > 1}
AUTOCORR = {'DADA pre-accident': (temporal_autocorrelation(dada_pre), DADA_STEP_S),
            'D2City V0': (temporal_autocorrelation(D2_FEATS['V0']), D2_STRIDE / D2_FPS)}
fig, ax = plt.subplots(figsize=(6, 4))
REPORT['autocorr'] = {}
for name, (ac, step) in AUTOCORR.items():
    lags = sorted(ac['cosine_by_lag'], key=lambda k: int(k[3:]))
    means = [ac['cosine_by_lag'][k]['mean'] for k in lags]
    REPORT['autocorr'][name] = {'step_s': step, 'cosine_by_lag': ac['cosine_by_lag']}
    xs = [int(k[3:]) * step for k in lags]
    ax.plot(xs, means, marker='o', label=f'{name} ({step:.3f} s/step)')
ax.set_xlabel('seconds between frames')
ax.set_ylabel('mean cosine(f_t, f_t+lag)')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'autocorr_seconds.png', dpi=110)
plt.show()
save_report()

## 4. Probes — **S**, S-ref, **R0**, **X**, **M** (plan §6 P5)

Every probe: standardized logistic regression, `C = EDA_PROBE_C`, balanced classes, folds grouped by
**source clip** (all segments of one D2City clip share a fold), and **the same folds for every probe**,
so Δ(X−R0) and Δ(M−R0) are paired per fold (t95 over 5 folds, 4 df).

- **R0** — DADA in-span = 1 vs DADA out-of-span (in-video) = 0 · reference ≈ Gate D0.
- **X** — DADA in-span = 1 vs **D2City only** = 0 · **the gate**.
- **M** — DADA in-span = 1 vs DADA out-of-span **+** D2City = 0 · the additive question.
- scored on held-out DADA videos: `auc_macro` (within-video), plus **shortcut AUC** =
  P(score(DADA normal frame) > score(D2City frame)) on held-out folds.
- **S** — DADA pre-accident = 1 vs D2City = 0 (pooled AUC). **S-ref** — DADA pre-accident vs DoTA
  pre-anomaly: how separable *any* other dashcam corpus is from DADA.

In [ ]:
import ctypes
import gc

import psutil
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from core.metrics import frame_auc

# Section 2 leaves the CLIP encoder, a CUDA context and PyAV decode buffers (3 decoder
# threads -> glibc arenas that never shrink) in this process. The probes need ~2 GB on top;
# on a 12.7 GB runtime that is the difference between finishing and a dead kernel.
for _name in ('ENCODER', 'frames', 'futures', 'mid', 'rows'):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
ctypes.CDLL('libc.so.6').malloc_trim(0)


def rss_gb() -> float:
    return psutil.Process().memory_info().rss / 2**30


print(f'RSS before probes: {rss_gb():.1f} GB of {psutil.virtual_memory().total / 2**30:.1f} GB')


def assign_folds(groups: list[str], k: int = FOLDS, seed: int = SEED) -> dict[str, int]:
    order = np.random.default_rng(seed).permutation(len(groups))
    return {groups[i]: int(pos % k) for pos, i in enumerate(order)}


def fit_score(pos: list[np.ndarray], neg: list[np.ndarray], tests: list[np.ndarray]) -> list[np.ndarray]:
    x = np.concatenate(pos + neg).astype(np.float64)
    y = np.concatenate([np.ones(sum(len(a) for a in pos)), np.zeros(sum(len(a) for a in neg))])
    # copy=False: standardize x in place -- same numbers (checked: max diff 0.0), one float64
    # matrix instead of two. Test arrays are astype() copies, so the cached features are untouched.
    scaler = StandardScaler(copy=False).fit(x)
    model = LogisticRegression(C=constants.EDA_PROBE_C, max_iter=constants.EDA_PROBE_MAX_ITER,
                               class_weight='balanced', random_state=SEED)
    model.fit(scaler.transform(x), y)
    del x
    return [model.decision_function(scaler.transform(t.astype(np.float64))) for t in tests]


def pooled_auc(a: list[np.ndarray], b: list[np.ndarray]) -> float:
    # P(score of a class-`a` row > score of a class-`b` row).
    a_, b_ = np.concatenate(a), np.concatenate(b)
    return frame_auc(np.concatenate([a_, b_]), np.concatenate([np.ones(len(a_)), np.zeros(len(b_))]))


def t95(deltas: list[float]) -> dict:
    d = np.asarray(deltas)
    half = T95 * d.std(ddof=1) / np.sqrt(len(d))
    return {'mean': float(d.mean()), 'lo': float(d.mean() - half), 'hi': float(d.mean() + half),
            'per_fold': d.tolist()}


DADA_IDS = sorted(DADA_FEATS)
DADA_FOLD = assign_folds(DADA_IDS)
SEGS = SEGMENTS[CHOSEN]
D2_FOLD = assign_folds(sorted({clip for _, clip, _, _ in SEGS}), seed=SEED + 1)
DOTA_FOLD = assign_folds(sorted(DOTA_PRE), seed=SEED + 2)
pos_rows = {v: DADA_FEATS[v][DADA_LABELS[v] == 1] for v in DADA_IDS}
neg_rows = {v: DADA_FEATS[v][DADA_LABELS[v] == 0] for v in DADA_IDS}
print(f'folds: DADA {len(DADA_IDS)} videos | D2City {len(D2_FOLD)} clips / {len(SEGS)} bags | DoTA {len(DOTA_PRE)}')

In [ ]:
def run_arm(arm: str) -> dict:
    feats = D2_FEATS[arm]
    seg_rows = [(clip, feats[clip][s:e]) for _, clip, s, e in SEGS]
    per_video = {p: {} for p in ('R0', 'X', 'M')}
    fold_macro = {p: [] for p in per_video}
    shortcut = {p: ([], []) for p in per_video}
    s_scores, sref_scores = ([], []), ([], [])
    for f in range(FOLDS):
        tr = [v for v in DADA_IDS if DADA_FOLD[v] != f]
        te = [v for v in DADA_IDS if DADA_FOLD[v] == f]
        d2_tr = [a for c, a in seg_rows if D2_FOLD[c] != f]
        d2_te = [a for c, a in seg_rows if D2_FOLD[c] == f]
        tests = [DADA_FEATS[v] for v in te] + d2_te
        negs = {'R0': [neg_rows[v] for v in tr], 'X': d2_tr, 'M': [neg_rows[v] for v in tr] + d2_tr}
        for name, neg in negs.items():
            scores = fit_score([pos_rows[v] for v in tr], neg, tests)
            dada_scores, d2_scores = scores[:len(te)], scores[len(te):]
            aucs = []
            for v, sc in zip(te, dada_scores):
                lab = DADA_LABELS[v]
                if 0 < lab.sum() < len(lab):
                    per_video[name][v] = frame_auc(sc, lab)
                    aucs.append(per_video[name][v])
            fold_macro[name].append(float(np.mean(aucs)))
            shortcut[name][0].extend(sc[DADA_LABELS[v] == 0] for v, sc in zip(te, dada_scores))
            shortcut[name][1].extend(d2_scores)
        # S: DADA pre-accident (1) vs D2City (0)
        pre_tr = [dada_pre[v] for v in tr if v in dada_pre]
        pre_te = [dada_pre[v] for v in te if v in dada_pre]
        sc = fit_score(pre_tr, d2_tr, pre_te + d2_te)
        s_scores[0].extend(sc[:len(pre_te)])
        s_scores[1].extend(sc[len(pre_te):])
        if arm == 'V0' and DOTA_PRE:   # S-ref does not depend on the D2City arm
            dota_tr = [a for v, a in DOTA_PRE.items() if DOTA_FOLD[v] != f]
            dota_te = [a for v, a in DOTA_PRE.items() if DOTA_FOLD[v] == f]
            sc = fit_score(pre_tr, dota_tr, pre_te + dota_te)
            sref_scores[0].extend(sc[:len(pre_te)])
            sref_scores[1].extend(sc[len(pre_te):])
        print(f'  {arm} fold {f}: ' + '  '.join(f'{p} {fold_macro[p][-1]:.4f}' for p in fold_macro)
              + f'  | RSS {rss_gb():.1f} GB', flush=True)
        gc.collect()
    out = {p: {'auc_macro': float(np.mean(list(per_video[p].values()))),
               'videos': len(per_video[p]), 'fold_macro': fold_macro[p],
               'shortcut_auc': pooled_auc(*shortcut[p])} for p in per_video}
    out['delta_X_R0'] = t95(np.subtract(fold_macro['X'], fold_macro['R0']).tolist())
    out['delta_M_R0'] = t95(np.subtract(fold_macro['M'], fold_macro['R0']).tolist())
    out['S'] = pooled_auc(*s_scores)
    if sref_scores[0]:
        out['S_ref_DoTA'] = pooled_auc(*sref_scores)
    return out


PROBES = {}
for arm in ('V0', 'V1'):
    # Resumable: a finished arm is on Drive; re-running this cell after a crash skips it.
    arm_path = OUT / f'probe_{arm}.json'
    if arm_path.is_file():
        PROBES[arm] = json.loads(arm_path.read_text())
        print(f'{arm} loaded from {arm_path.name}')
        continue
    t0 = time.time()
    PROBES[arm] = run_arm(arm)
    arm_path.write_text(json.dumps(PROBES[arm], indent=2, default=float))
    print(f'{arm} done in {(time.time() - t0) / 60:.1f} min')
PROBES['V1']['S_ref_DoTA'] = PROBES['V0'].get('S_ref_DoTA')
PROBES['V0'].setdefault('S_ref_DoTA', None)
REPORT['probes'] = PROBES
save_report()

## 5. Verdict (plan §6 P6) — read against the table, not against hope

In [ ]:
def verdict(p: dict) -> dict:
    r0, x, m = p['R0']['auc_macro'], p['X']['auc_macro'], p['M']['auc_macro']
    dx, dm = p['delta_X_R0']['mean'], p['delta_M_R0']['mean']
    sane = R0_SANITY[0] <= r0 <= R0_SANITY[1]
    if x < GX_FAIL or dx < GX_DELTA_MARGINAL:
        gx = 'FAIL'
    elif x >= GX_PASS and dx >= GX_DELTA_PASS:
        gx = 'PASS'
    else:
        gx = 'MARGINAL'
    gm = dm >= GM_DELTA
    if not sane:
        call = 'PIPELINE BUG -- R0 outside the sanity band; do not read X/M'
    elif gx == 'FAIL':
        call = 'NO-GO -- D2City teaches source, not accident'
    elif not gm:
        call = 'NO-GO (additive) -- adding D2City costs localization; stay on T2'
    elif gx == 'MARGINAL':
        call = 'GO-with-in-video-negatives (M-style corpus only)'
    else:
        call = 'GO -- write the corpus-build plan'
    return {'R0': r0, 'X': x, 'M': m, 'dX': p['delta_X_R0'], 'dM': p['delta_M_R0'], 'R0_sane': sane,
            'G-X': gx, 'G-M': gm, 'S': p['S'], 'S_ref': p['S_ref_DoTA'],
            'S_vs_ref': ('n/a (no DoTA)' if p['S_ref_DoTA'] is None
                         else 'no more foreign than DoTA' if p['S'] <= p['S_ref_DoTA'] + GS_REF_MARGIN
                         else 'MORE foreign than DoTA'),
            'shortcut_X': p['X']['shortcut_auc'], 'shortcut_M': p['M']['shortcut_auc'],
            'shortcut_red_flag': max(p['X']['shortcut_auc'], p['M']['shortcut_auc']) > SHORTCUT_RED,
            'call': call}


VERDICT = {arm: verdict(PROBES[arm]) for arm in ('V0', 'V1')}
v0, v1 = VERDICT['V0'], VERDICT['V1']
prefer_v1 = (v1['X'] - v0['X'] >= ARM_RULE_MARGIN) or (v0['S'] - v1['S'] >= ARM_RULE_MARGIN and v1['X'] >= v0['X'])
ARM = 'V1' if prefer_v1 else 'V0'
REPORT['verdict'] = {'per_arm': VERDICT, 'arm': ARM, 'call': VERDICT[ARM]['call'],
                     'G-I': REPORT['G-I']['pass'], 'G-L': REPORT['G-L']['pass']}
save_report()

fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
ci = lambda d: f"{d['mean']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]"
lines = ['# D2City normal-bag EDA -- read-out', '',
         f"G-I {'PASS' if REPORT['G-I']['pass'] else 'FAIL'} | G-L ({CHOSEN}) length AUC "
         f"{REPORT['G-L']['length_auc_mix1']:.4f} {'PASS' if REPORT['G-L']['pass'] else 'FAIL'}", '',
         '| quantity | V0 | V1 |', '|---|---:|---:|']
for key, label in (('R0', 'R0 auc_macro'), ('X', 'X auc_macro'), ('M', 'M auc_macro'),
                   ('S', 'S (DADA-pre vs D2City)'), ('S_ref', 'S-ref (DADA-pre vs DoTA)'),
                   ('shortcut_X', 'shortcut AUC, X'), ('shortcut_M', 'shortcut AUC, M'),
                   ('G-X', 'G-X'), ('G-M', 'G-M'), ('S_vs_ref', 'S vs S-ref')):
    lines.append(f'| {label} | {fmt(v0[key])} | {fmt(v1[key])} |')
lines.append(f"| Δ(X−R0) t95 | {ci(v0['dX'])} | {ci(v1['dX'])} |")
lines.append(f"| Δ(M−R0) t95 | {ci(v0['dM'])} | {ci(v1['dM'])} |")
lines += ['', f'**Arm:** {ARM}  ·  **Call:** {VERDICT[ARM]["call"]}']
(OUT / 'eda_d2city.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
print(f'\nwritten: {OUT}/eda_d2city.json, eda_d2city.md, montage.png, autocorr_seconds.png, inventory.csv')